In [1]:
import pandas as pd
import numpy as np


In [2]:
# 1. Load the dataset generated in stage 03
try:
    df = pd.read_csv('data/edreams_stage3_data.csv')
    print(f"Success! Dataset loaded with {df.shape[0]} rows.")
except FileNotFoundError:
    print("Error: The file 'data/edreams_stage3_data.csv' was not found. Please ensure you ran script 03 first.") 

Success! Dataset loaded with 1000 rows.


## calculate_accessibility_roi

In [3]:
# 2. Create the Accessibility analysis tool
def calculate_accessibility_roi(df: pd.DataFrame) -> str:
    """
    Calculate the baggage upsell conversion rate for visually impaired users (PCD)
    before and after the layout bug fix, estimating the financial impact.
    """
    # Filter only visually impaired users
    df_pcd = df[df['is_visually_impaired']==1]
    
    if df_pcd.empty:
        return "No data found for visually impaired users to analyze."
    
    # Group by bug status (1 = active/before fix, 0 = inactive/after fix)
    summary = df_pcd.groupby('layout_bug_active')['baggage_upsell_purchased'].agg(['count', 'sum', 'mean']).reset_index()
                                 
    try:
        conv_with_bug = summary[summary['layout_bug_active'] == 1]['mean'].values[0]
        conv_fixed = summary[summary['layout_bug_active'] == 0]['mean'].values[0]
        total_pcd_post_fix = summary[summary['layout_bug_active'] == 0]['count'].values[0]
        
    except IndexError:
        return "Insufficient data to calculate ROI."
    
    baggage_price = 30.0
        
    # Calculate conversion lift and recovered revenue
    conversion_impact  = conv_fixed - conv_with_bug
    recovered_sales = conversion_impact * total_pcd_post_fix
    recovered_revenue = recovered_sales * baggage_price

    # Format response for the AI Agent
    result = (
        f"--- ROI ANALYSIS: ACCESSIBILITY ---\n"
        f"Baggage conversion rate WITH BUG: {conv_with_bug*100:.2f}%\n"
        f"Baggage conversion rate AFTER FIX: {conv_fixed*100:.2f}%\n"
        f"Absolute conversion lift: +{conversion_impact*100:.2f}%\n"
        f"Estimated recovered revenue: €{recovered_revenue:.2f}\n"
    )
    return result
    
# Locally test the tool
print(calculate_accessibility_roi(df))
        
    

--- ROI ANALYSIS: ACCESSIBILITY ---
Baggage conversion rate WITH BUG: 5.41%
Baggage conversion rate AFTER FIX: 30.56%
Absolute conversion lift: +25.15%
Estimated recovered revenue: €271.62



## optimize_seat_pricing_roi

In [4]:
# 3. Create the Seat Pricing optimization tool
def optimize_seat_pricing_roi(df: pd.DataFrame) -> str:
    """
    Compares seat upsell revenue and purchase volumn between the fixed pricing model and the dynamic pricing model.
    """
    if df.empty:
        return "No data available for pricing analysis."
    
    # Create a helper column: 1 if seat was purchased (price > 0), 0 otherwise
    df['seat_purchased'] = (df['seat_price_paid'] > 0).astype(int)
    
    # Group by pricing model to calculate total revenue, volume, and average paid
    summary = df.groupby('layout_bug_active')\
                ['seat_price_paid']\
                .agg(['sum', 'count','mean'])\
                .reset_index()
    
    # Group by pricing model to calculate conversion rate (purchase rate)
    conversion_summary = df.groupby('layout_bug_active')\
                            ['seat_purchased']\
                            .mean()\
                            .reset_index()
    
    try:
        # Isolate baseline data (Fixed Pricing = 0)
        revenue_fixed = summary[summary['layout_bug_active'] == 0]['sum'].values[0]
        conv_fixed = conversion_summary[conversion_summary['layout_bug_active'] == 0]['seat_purchased'].values[0]
        
        # Isolate variant data (Dynamic Pricing = 1)                                
        revenue_dynamic = summary[summary['layout_bug_active'] == 1]['sum'].values[0] 
        conv_dynamic = conversion_summary[conversion_summary['layout_bug_active'] == 1]['seat_purchased'].values[0]
        
     
        
    except IndexError:
        return "Insufficient data to compare pricing models."
                                        
    # Calculate performance metrics (deltas)
    revenue_lift = revenue_dynamic - revenue_fixed
    conversion_impact = conv_dynamic - conv_fixed
                                        
    # Format executive report for the AI Agent
    result = (
        f"--- ROI ANALYSIS: SEAT DYNAMIC PRICING ---\n"
        f"Fixed Pricing - Conversion:{conv_fixed*100:.2f}%\n"
        f"Total Revenue:€{revenue_fixed:.2f}\n"
        f"Dynamic Pricing - Conversion:{conv_dynamic*100:.2f}%\n"
        f"Total Revenue:€{revenue_dynamic:.2f}\n"
        f"Absolute Revenue Lift (Dynamic vs Fixed):€{revenue_lift:+.2f}\n"
    )
    return result

# Locally test the tool
print(optimize_seat_pricing_roi(df))

--- ROI ANALYSIS: SEAT DYNAMIC PRICING ---
Fixed Pricing - Conversion:100.00%
Total Revenue:€34750.15
Dynamic Pricing - Conversion:100.00%
Total Revenue:€40128.02
Absolute Revenue Lift (Dynamic vs Fixed):€+5377.87



In [5]:
df.columns

Index(['transaction_id', 'route', 'seat_type', 'flight_occupancy_pct',
       'seat_price_paid', 'is_visually_impaired', 'layout_bug_active',
       'baggage_upsell_purchased', 'uses_postal_api', 'payment_declined',
       'seat_purchased'],
      dtype='object')

### Create a realistic dataset

In [6]:
# 1. Split baseline (Normal Layout = 0) and variant (Buggy Layout = 1)
group_normal = df[df['layout_bug_active'] == 0]
group_buggy = df[df['layout_bug_active'] == 1]

In [7]:
# 2. Calculate dropout rows needed to simulate a realistic conversion rate
# To drop baseline conversion to ~40%, we need 1.5x more dropout rows
n_dropouts_normal = int(len(group_normal) * 1.5)
# To drop variant conversion to ~35%, we need 1.85x more dropout rows
n_dropouts_buggy = int(len(group_buggy) * 1.85)

In [8]:
# 3. Create simulated user rows who chose not to purchase (seat_price_paid = 0.0)
dropouts_normal = pd.DataFrame({
    'layout_bug_active':[0]*n_dropouts_normal,
    'seat_price_paid':[0.0]*n_dropouts_normal, 
    'transaction_id':[f"sim_dropout_0_{i}" for i in range(n_dropouts_normal)]
})
                       
dropouts_buggy = pd.DataFrame({
    'layout_bug_active': [1] * n_dropouts_buggy,
    'seat_price_paid': [0.0] * n_dropouts_buggy,
    'transaction_id': [f"sim_dropout_1_{i}" for i in range(n_dropouts_buggy)]
})

In [9]:
# dropouts_buggy

In [10]:
# dropouts_normal

In [11]:
# 4. Merge original data with the simulated dropout data
df_realistic = pd.concat([df, dropouts_normal, dropouts_buggy], ignore_index=True)

In [12]:
df_realistic

,transaction_id,route,seat_type,flight_occupancy_pct,seat_price_paid,is_visually_impaired,layout_bug_active,baggage_upsell_purchased,uses_postal_api,payment_declined,seat_purchased
0,1000,bcn-cdg,window,96.0,84.20,0.0,1,0.0,0.0,0.0,1.0
1,1001,bcn-cdg,middle,61.0,54.39,0.0,0,0.0,1.0,0.0,1.0
2,1002,bcn-ory,aisle,65.0,78.00,1.0,0,0.0,0.0,0.0,1.0
3,1003,bcn-cdg,aisle,73.0,79.60,0.0,1,0.0,1.0,0.0,1.0
4,1004,bcn-ory,window,68.0,78.60,0.0,1,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2681,sim_dropout_1_982,NaN,NaN,NaN,0.00,NaN,1,NaN,NaN,NaN,NaN
2682,sim_dropout_1_983,NaN,NaN,NaN,0.00,NaN,1,NaN,NaN,NaN,NaN
2683,sim_dropout_1_984,NaN,NaN,NaN,0.00,NaN,1,NaN,NaN,NaN,NaN
2684,sim_dropout_1_985,NaN,NaN,NaN,0.00,NaN,1,NaN,NaN,NaN,NaN


In [13]:
# # 5. Fill missing values for the newly created columns with 0
# df_realistic = df_realistic.fillna(0)
# df_realistic

In [14]:
# 5. Clean missing values using professional data standards (Strings for text, floats for numbers)
df_realistic['route'] = df_realistic['route'].fillna('Unknown')
df_realistic['seat_type'] = df_realistic['seat_type'].fillna('Not Selected')

# For numerical columns where missing means zero (like occupancy or metrics), we keep 0
df_realistic['flight_occupancy_pct'] = df_realistic['flight_occupancy_pct'].fillna(0.0)
df_realistic['is_visually_impaired'] = df_realistic['is_visually_impaired'].fillna(0)
df_realistic['baggage_upsell_purchased'] = df_realistic['baggage_upsell_purchased'].fillna(0)
df_realistic['uses_postal_api'] = df_realistic['uses_postal_api'].fillna(0)
df_realistic['payment_declined'] = df_realistic['payment_declined'].fillna(0)

df_realistic

,transaction_id,route,seat_type,flight_occupancy_pct,seat_price_paid,is_visually_impaired,layout_bug_active,baggage_upsell_purchased,uses_postal_api,payment_declined,seat_purchased
0,1000,bcn-cdg,window,96.0,84.20,0.0,1,0.0,0.0,0.0,1.0
1,1001,bcn-cdg,middle,61.0,54.39,0.0,0,0.0,1.0,0.0,1.0
2,1002,bcn-ory,aisle,65.0,78.00,1.0,0,0.0,0.0,0.0,1.0
3,1003,bcn-cdg,aisle,73.0,79.60,0.0,1,0.0,1.0,0.0,1.0
4,1004,bcn-ory,window,68.0,78.60,0.0,1,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2681,sim_dropout_1_982,Unknown,Not Selected,0.0,0.00,0.0,1,0.0,0.0,0.0,NaN
2682,sim_dropout_1_983,Unknown,Not Selected,0.0,0.00,0.0,1,0.0,0.0,0.0,NaN
2683,sim_dropout_1_984,Unknown,Not Selected,0.0,0.00,0.0,1,0.0,0.0,0.0,NaN
2684,sim_dropout_1_985,Unknown,Not Selected,0.0,0.00,0.0,1,0.0,0.0,0.0,NaN


In [15]:
# Test your ROI tool using the newly cleaned professional dataset
print(optimize_seat_pricing_roi(df_realistic))

--- ROI ANALYSIS: SEAT DYNAMIC PRICING ---
Fixed Pricing - Conversion:40.00%
Total Revenue:€34750.15
Dynamic Pricing - Conversion:35.11%
Total Revenue:€40128.02
Absolute Revenue Lift (Dynamic vs Fixed):€+5377.87



## evaluate_payment_fraud_risk